<a href="https://colab.research.google.com/github/areslou/cpecog2-coral-bleaching/blob/main/notebooks/Group2_Coral_Run5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Group 2 CPECOG2: Coral bleaching, Run 5

Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all.

Run 5 is a focused augmentation experiment based on the Run 3 final configuration: YOLOv8s-cls, dropout 0.3, the 629/132/131 split, and the seed-42 validation threshold.

The sweep varies only scale/crop-oriented augmentation and compares four candidates on seed-42 validation data:

- R12 Run 3 reference
- R13 scale 0.1 plus random erase
- R14 scale 0.3 plus translate 0.1
- R15 scale 0.3 plus translate 0.1 and random erase

The held-out test set and external BU/BHD set are evaluated only after a candidate and threshold are selected. The selected threshold must achieve at least 0.90 validation bleached recall.

Outputs go to `results/run5/` and are zipped as `run5_results.zip`.

In [ ]:
!pip -q install ultralytics==8.4.211 pi-heif imagehash kagglehub huggingface_hub

In [ ]:
import os, glob, shutil, json, zipfile, time, subprocess
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import f1_score, recall_score
from ultralytics import YOLO

SEED = 42
FINAL_SEEDS = [42, 43, 44]
IMGSZ = 224
BATCH = 32
EPOCHS = 30
PATIENCE = 8
CLASSES = ["healthy", "bleached"]
SPLITS_URL = "https://raw.githubusercontent.com/areslou/cpecog2-coral-bleaching/main/data/splits.csv"

import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

if os.path.isdir("/content"):
    REPO = "/content/cpecog2-coral-bleaching"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "https://github.com/areslou/cpecog2-coral-bleaching.git", REPO], check=True)
    os.chdir(REPO)
    BASE = REPO
else:
    BASE = os.getcwd()
OUT = os.path.join(BASE, "results", "run5")
os.makedirs(OUT, exist_ok=True)

In [ ]:
import kagglehub
import imagehash

DATA_SRC = kagglehub.dataset_download("vencerlanz09/healthy-and-bleached-corals-image-classification")
splits = pd.read_csv(SPLITS_URL)
splits["file"] = splits["path"].map(os.path.basename)

paths = sorted(p for p in glob.glob(os.path.join(DATA_SRC, "**", "*"), recursive=True)
               if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")))
def label_of(p):
    parent = os.path.basename(os.path.dirname(p)).lower()
    if "bleach" in parent:
        return "bleached"
    if "health" in parent:
        return "healthy"
    return None

def phash_of(p):
    try:
        with Image.open(p) as im:
            return str(imagehash.phash(im.convert("RGB")))
    except Exception:
        return None

df = pd.DataFrame({"path": paths})
df["file"] = df["path"].map(os.path.basename)
df["label"] = df["path"].map(label_of)
df["phash"] = df["path"].map(phash_of)
df = df.merge(splits[["file", "label", "split", "phash"]], on=["file", "label"], how="inner", suffixes=("", "_fixed"))
df["phash"] = df["phash_fixed"]
df = df.dropna(subset=["split", "phash_fixed"]).reset_index(drop=True)
print(pd.crosstab(df["split"], df["label"], margins=True))
df[["file", "label", "split", "phash"]].rename(columns={"file": "path"}).to_csv(os.path.join(OUT, "splits_v2.csv"), index=False)

In [ ]:
ROOT = os.path.join(BASE, "coral_run5")
if os.path.exists(ROOT):
    shutil.rmtree(ROOT)
for split in ("train", "val", "test"):
    for label in CLASSES:
        os.makedirs(os.path.join(ROOT, split, label), exist_ok=True)

def pad_square(p):
    im = Image.open(p).convert("RGB")
    s = max(im.size)
    sq = Image.new("RGB", (s, s), (0, 0, 0))
    sq.paste(im, ((s - im.size[0]) // 2, (s - im.size[1]) // 2))
    return sq

for r in df.itertuples():
    dst = os.path.join(ROOT, r.split, r.label, r.file)
    pad_square(r.path).save(dst, quality=95)
print("Prepared:", {s: sum(len(glob.glob(os.path.join(ROOT, s, "*", "*"))) for s in ["train", "val", "test"] if s == s) for s in ("train", "val", "test")})

In [ ]:
CANDIDATES = {
    "R12_run3_reference": dict(scale=0.1, erasing=0.0, translate=0.0),
    "R13_scale_randomerase": dict(scale=0.25, erasing=0.2, translate=0.0),
    "R14_scale_translate": dict(scale=0.25, erasing=0.0, translate=0.1),
    "R15_scale_translate_randomerase": dict(scale=0.25, erasing=0.2, translate=0.1),
}

VAL_FILES = glob.glob(os.path.join(ROOT, "val", "*", "*"))
y_val = np.array([1 if os.path.basename(os.path.dirname(p)) == "bleached" else 0 for p in VAL_FILES])
sweep = []

for name, aug in CANDIDATES.items():
    run_dir = os.path.join(BASE, "run5_sweep", name)
    model = YOLO("yolov8s-cls.pt")
    model.train(data=ROOT, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE, seed=SEED,
                project=os.path.dirname(run_dir), name=os.path.basename(run_dir), exist_ok=True,
                dropout=0.3, patience=PATIENCE, pretrained=True,
                auto_augment=None, hsv_h=0.0, hsv_s=0.2, hsv_v=0.1,
                fliplr=0.5, flipud=0.5,
                **aug)
    trained = YOLO(os.path.join(run_dir, "weights", "best.pt"))
    bleached_index = next(index for index, name in trained.names.items() if name == "bleached")
    probs = np.array([float(r.probs.data[bleached_index]) for r in trained.predict(VAL_FILES, imgsz=IMGSZ, verbose=False)])
    macro_f1 = []
    for threshold in np.linspace(0.01, 0.99, 981):
        pred = probs >= threshold
        recall = recall_score(y_val, pred, zero_division=0)
        macro_f1.append((f1_score(y_val, pred, average="macro"), recall, threshold))
    eligible = [r for r in macro_f1 if r[1] >= 0.90]
    if not eligible:
        sweep.append(dict(name=name, validation_macro_f1=float("nan"), validation_bleached_recall=max(r[1] for r in macro_f1), threshold=None, settings=aug))
        continue
    best = max(eligible, key=lambda r: r[0])
    sweep.append(dict(name=name, validation_macro_f1=best[0], validation_bleached_recall=best[1], threshold=best[2], settings=aug))
    pd.DataFrame(dict(path=VAL_FILES, true_label=y_val, p_bleached=probs)).to_csv(os.path.join(OUT, name + "_val_predictions.csv"), index=False)

pd.DataFrame(sweep).sort_values("validation_macro_f1", ascending=False).to_csv(os.path.join(OUT, "sweep_results.csv"), index=False)
print(pd.DataFrame(sweep).sort_values("validation_macro_f1", ascending=False).to_string(index=False))

In [ ]:
sweep_df = pd.read_csv(os.path.join(OUT, "sweep_results.csv"))
selected = sweep_df.sort_values(["validation_macro_f1", "validation_bleached_recall"], ascending=False).iloc[0]
selected_name = selected["name"]
threshold = float(selected["threshold"])
selected_aug = CANDIDATES[selected_name]
with open(os.path.join(OUT, "selected_candidate.json"), "w") as f:
    json.dump(dict(name=selected_name, threshold=threshold, settings=selected_aug), f, indent=2)
pd.DataFrame([dict(threshold=threshold, source="seed42_validation_only")]).to_csv(os.path.join(OUT, "selected_threshold.csv"), index=False)
print("Selected:", selected_name)
print("Validation-selected threshold:", threshold)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score

rows = []
for seed in FINAL_SEEDS:
    run_dir = os.path.join(BASE, "run5_final", f"seed{seed}")
    model = YOLO("yolov8s-cls.pt")
    model.train(data=ROOT, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE, seed=seed,
                project=os.path.dirname(run_dir), name=os.path.basename(run_dir), exist_ok=True,
                dropout=0.3, patience=PATIENCE, pretrained=True,
                auto_augment=None, hsv_h=0.0, hsv_s=0.2, hsv_v=0.1,
                fliplr=0.5, flipud=0.5,
                **selected_aug)
    final = YOLO(os.path.join(run_dir, "weights", "best.pt"))
    test_files = glob.glob(os.path.join(ROOT, "test", "*", "*"))
    y_test = np.array([1 if os.path.basename(os.path.dirname(p)) == "bleached" else 0 for p in test_files])
    bleached_index = next(index for index, name in final.names.items() if name == "bleached")
    p_test = np.array([float(r.probs.data[bleached_index]) for r in final.predict(test_files, imgsz=IMGSZ, verbose=False)])
    pred = p_test >= threshold
    rows.append(dict(seed=seed,
                     macro_f1=f1_score(y_test, pred, average="macro"),
                     bleached_recall=recall_score(y_test, pred, zero_division=0),
                     accuracy=accuracy_score(y_test, pred),
                     roc_auc=roc_auc_score(y_test, p_test),
                     confusion_matrix=confusion_matrix(y_test, pred, labels=[0, 1]).tolist()))
    shutil.copy(os.path.join(run_dir, "weights", "best.pt"), os.path.join(OUT, f"final_model_s{seed}.pt"))
    pd.DataFrame(dict(path=test_files, true_label=y_test, p_bleached=p_test)).to_csv(os.path.join(OUT, f"test_predictions_seed{seed}.csv"), index=False)

pd.DataFrame(rows).to_csv(os.path.join(OUT, "test_results.csv"), index=False)
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# External evaluation: run only after the candidate and threshold are fixed.
# Reuse eval_external.py with the Run 5 seed models and the fixed validation-selected threshold.
weights = " ".join(os.path.join(OUT, f"final_model_s{seed}.pt") for seed in FINAL_SEEDS)
!python {BASE}/eval_external.py --weights {weights} --out {OUT} --threshold {threshold}

In [ ]:
summary = {
    "run": 5,
    "selected_candidate": selected_name,
    "selected_settings": selected_aug,
    "threshold": threshold,
    "threshold_source": "seed42_validation_only",
    "test_results": rows,
}
with open(os.path.join(OUT, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

shutil.make_archive(os.path.join(BASE, "run5_results"), "zip", OUT)
print("Saved", os.path.join(BASE, "run5_results.zip"))